In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(os.getcwd())

c:\Users\Javed\Fake-News-Headline-Detector


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib, json
from src.clusters import TopicClusterer

df = pd.read_csv('data/headlines_clean.csv')
print(df['label'].unique())

clusterer = TopicClusterer().fit(df['title'])
print('Chosen k:', clusterer.k)
print(clusterer.scores)

Matplotlib is building the font cache; this may take a moment.


<ArrowStringArray>
['fake', 'real']
Length: 2, dtype: str
Chosen k: 12
{4: 0.004468221621692384, 5: 0.004828904873847916, 6: 0.005304341048175897, 7: 0.006340663659260731, 8: 0.006935673391853415, 9: 0.00700926718928531, 10: 0.007833917356837444, 11: 0.008212522414294817, 12: 0.00836987209999063}


In [3]:
for c, terms in clusterer.top_terms().items():
    print(c, terms)

0 ['trump', 'president', 'court', 'republican', 'state', 'donald', 'election', 'police']
1 ['says', 'trump', 'china', 'pm', 'iran', 'brexit', 'deal', 'turkey']
2 ['new', 'york', 'trump', 'zealand', 'jersey', 'times', 'says', 'shows']
3 ['russia', 'probe', 'trump', 'says', 'sanctions', 'syria', 'putin', 'election']
4 ['clinton', 'hillary', 'trump', 'fbi', 'campaign', 'emails', 'email', 'sanders']
5 ['obama', 'trump', 'president', 'michelle', 'says', 'court', 'america', 'barack']
6 ['mexico', 'border', 'wall', 'trump', 'says', 'street', 'tax', 'irish']
7 ['myanmar', 'rohingya', 'bangladesh', 'violence', 'refugees', 'crisis', 'journalists', 'says']
8 ['korea', 'north', 'south', 'says', 'china', 'sanctions', 'nuclear', 'japan']
9 ['just', 'trump', 'got', 'obama', 'donald', 'said', 'twitter', 'hillary']
10 ['senate', 'panel', 'tax', 'healthcare', 'trump', 'republican', 'vote', 'republicans']
11 ['house', 'white', 'trump', 'speaker', 'tax', 'ryan', 'says', 'panel']


In [5]:
names = {
    0: 'Courts, Police and Elections',
    1: 'Trump and World Diplomacy',
    2: 'New York and New Zealand',
    3: 'Russia Probe and Sanctions',
    4: 'Clinton Emails and FBI',
    5: 'Obama',
    6: 'Border Wall and Tax',
    7: 'Rohingya Crisis',
    8: 'North Korea',
    9: 'Twitter and Trump Reactions',
    10: 'Senate Healthcare and Tax',
    11: 'House and Speaker Ryan',
}
print(names)

{0: 'Courts, Police and Elections', 1: 'Trump and World Diplomacy', 2: 'New York and New Zealand', 3: 'Russia Probe and Sanctions', 4: 'Clinton Emails and FBI', 5: 'Obama', 6: 'Border Wall and Tax', 7: 'Rohingya Crisis', 8: 'North Korea', 9: 'Twitter and Trump Reactions', 10: 'Senate Healthcare and Tax', 11: 'House and Speaker Ryan'}


In [ ]:
df['cluster'] = clusterer.predict(df['title'])
summary = pd.DataFrame({
    'name': pd.Series(names),
    'size': df['cluster'].value_counts().sort_index(),
    'fake_share': (df['label'] == 'fake').groupby(df['cluster']).mean().round(3),
})
print(summary)

In [ ]:
os.makedirs('figures', exist_ok=True)
plt.figure(figsize=(9, 5))
plt.barh([names[c] for c in summary.index], summary['size'])
plt.xlabel('Number of headlines')
plt.title('Headline topic clusters')
plt.savefig('figures/clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save the clusterer and the names
joblib.dump(clusterer, 'models/clusterer.joblib')
with open('models/cluster_names.json', 'w') as f:
    json.dump({str(k): v for k, v in names.items()}, f)
print('saved')